# Многорукие бандиты
#### Небольшое замечание

Итоговая энергия вычисляется как $E(s) = -\sum\limits_{(i,j) \in E} J_{ij} \cdot s_i \cdot s_j$. В лидерборде эта энергия умножается на -1, соответственно, знак минуса в формуле энергии исчезает. Поэтому в дальнейшем будем говорить о том, что мы максимизируем энергию, а все функции подсчёта энергии конфигурации считают эту энергию без минуса (как в лидерборде)

#### Основные идеи
Относительно нашей задачи можно сделать следующее справедливое замечание: у среды всего одно состояние, ведь граф вне зависимости от спинов фиксирован. Тогда попробуем решить нашу задачу при помощи алгоритма обучения с подкреплением - нестационарных многоруких бандитов.

Определим ручку как смену $s_i$ спина на $-s_i$, а за награду определим повышение энергии. Справедливо утверждение, что если мы в $k$ ход дёрнули ручку $s_i$ и получили награду $r_i$, то дёрнув эту же ручку в следующий ход мы получим награду $-r_i$.

Тогда определим стратегию агента следующим образом, каждый ход он будет дёргать ручку с наименьшей ожидаемой наградой, то есть ручку с номером $argmin(av)$, где $av$ - список с ожидаемыми наградами для каждой ручки. Ведь если в некоторый ход агент дёрнул за ручку и получил отрицательную награду, то в следующий ход, дёрнув за ту же ручку, мы получим положительную.

При инициализации агента он дёрнет за каждую ручку по два раза. Сначала он дёргает за ручку $i$, получает награду $r_i$, а после снова дёргает за эту же ручку, чтобы вернуть спины в начальное положение. В агента мы запишем награду $-r_i$, ведь дёрнув за неё в последующие ходы он получит предполагаемую награду $r_i$

#### Стратегия агента

Основная идея, описанная выше, агент дёргает за ручку с наименьшей ожидаемой наградой. Изначально $av_i = 0$ для любого $i$, обновление $av_i$ будет происходить по следующей формуле (как в нестационарных бандитах): $av_i += \alpha \cdot (r_i - av_i) $, где $\alpha$ некоторое число от 0 до 1, a $r_i$ некоторая полученная награда.

Однако, высока вероятность, что агент загонит себя в "локальный максимум" - конфигурацию спинов, из которую он не выйдет таким методом, и которая не является оптимальной. Также логично предположить, что не все выгодные ходы приведут в глобальный максимум. Для этого мы введем вероятность $\varepsilon$ - в каждый ход с такой вероятностью агент будет дёргать случайную ручку.

В ходе обучения и работы агента была выявлена следующая проблема: при нормальных значениях $\varepsilon$ (таких, что агент не совершает больше половины действий случайно), ему не хватает "рандома", чтобы выбраться из локального максимума и продолжить исследование дальше. Данная проблема была решена следующим образом: агент хранит 6 последних полученных энергий (историю). Если все энергии в истории совпадают с энергией на некотором ходу, то $\varepsilon$ делится на значение $0 < upEps < 1$. В ином случае $\varepsilon$ умножается на $0<lessEps < 1$ и $\varepsilon$ присваивается следующее значение: $\varepsilon = max(startEps,\varepsilon)$, где $startEps$ начальное значение $\varepsilon$.

Значения $\alpha, upEps, lessEps, startEps$ будут являться гиперпараметрами нашего агента.

In [1]:
import numpy as np
import pandas as pd
from random import *
from math import *
from copy import copy
from tqdm import *
seed(42)

In [2]:
#класс среды
class Env:
    def __init__(self, matrix):
        seed(42)
        self.mat = matrix
                            
        self.spins = np.array([choice([-1,1]) for i in range(50)])
        self.en = self.spins @ self.mat @ self.spins / 2
        self.bestSpins = copy(self.spins)
        self.maxEn = self.en
    
    def getScore(self):
        return self.en
    
    def action(self, action):
        oldEn = self.en
        
        # изменяем спин и обновляем энергию системы
        self.en -= 2 * self.spins[action] * (self.spins @ self.mat[action])
        self.spins[action] = -self.spins[action]
        
        
        # сохраняем лучшую конфигурацию спинов
        if self.en > self.maxEn:
            self.maxEn = self.en
            self.bestSpins = copy(self.spins)
        
        # как награда увеличение энергии
        return self.en - oldEn

In [3]:
# класс агента
class BanditHistory:
    def __init__(self, alpha =  0.8, eps = 0, lessEps = 0.99, upEps = 0.3):
        seed(42)
        self.av = np.zeros(50)
        self.exp = 0 # переменная-счётчик, чтобы в начале дёрнуть каждую ручку поочередно
        self.alpha = alpha
        self.eps = eps
        self.startEps = eps
        self.upEps = upEps
        self.lessEps = lessEps
        self.history = []
    
    def reward(self, act, rew):
        self.av[act] += self.alpha * (rew - self.av[act])
    
    def correctEps(self,nowScore):
        # корректируем эпсилон в зависимости от истории
        if self.history.count(nowScore) == 6:
            self.eps/=self.upEps
            self.eps = min(1, self.eps) # вероятность не может быть больше 1
        else:
            self.eps*=self.lessEps
            self.eps = max(self.eps, self.startEps)
            
        # Храним 6 последних энергий
        self.history.append(nowScore)
        if len(self.history) > 6:
            self.history = self.history[1:]

    def action(self):
        # первые 50 ходов поочерёдно дергаем каждую ручку
        if self.exp < 50:
            self.exp+=1
            return self.exp - 1
        
        if random() < self.eps:
            return randint(0,49)
        
        return np.argmin(self.av)

В ходе перебора гиперпараметров (вне этого блокнота), мы решили взять следующие $\alpha  = 0.3, \; \varepsilon = 0.35, \; lessEps = 0.8, \; upEps = 0.6$

In [4]:
m = np.load("test_matrices.npy") 

In [5]:
spins = [] # спины для каждой матрицы
myEns = [] # максимально полученная энергия для каждой матрицы
for j in range(60):
    env = Env(m[j])
    ag = BanditHistory(alpha = 0.3, eps = 0.35, lessEps = 0.8, upEps = 0.6)
    
    # Первые 50 действий - дергаем за каждую ручку дважды и записываем отрицательную награду
    for i in range(50):
        act = ag.action()
        rew = env.action(act)
        ag.reward(act,-rew) 
        env.action(act)
    
    mxScore = -10000 # максимальная энергия
    for i in tqdm(range(10**5)):
        act = ag.action() # агент совершает действие
        rew = env.action(act) # среда отдаёт награду
        ag.reward(act,rew) # агент получает награду и корректирует av
        
        nowScore = env.getScore()
        ag.correctEps(nowScore) # записываем энергию в историю и корректируем эпсилон
        
        mxScore = max(nowScore,mxScore)
    print(f'Матрица {j+1}, итоговая энергия: {mxScore}')
    
    spins.append(env.bestSpins)
    myEns.append(mxScore)

100%|███████████████████████████████████████████████████████████████████████| 100000/100000 [00:02<00:00, 49090.10it/s]


Матрица 1, итоговая энергия: 218.0


100%|███████████████████████████████████████████████████████████████████████| 100000/100000 [00:02<00:00, 48459.21it/s]


Матрица 2, итоговая энергия: 203.0


100%|███████████████████████████████████████████████████████████████████████| 100000/100000 [00:02<00:00, 46956.82it/s]


...

Матрица 58, итоговая энергия: 216.0


100%|███████████████████████████████████████████████████████████████████████| 100000/100000 [00:02<00:00, 48356.28it/s]


Матрица 59, итоговая энергия: 204.0


100%|███████████████████████████████████████████████████████████████████████| 100000/100000 [00:02<00:00, 49434.33it/s]

Матрица 60, итоговая энергия: 208.0


In [6]:
print('Итоговая метрика:', sum(myEns)/len(myEns))

Итоговая метрика: 205.95


In [7]:
# Экспорт спинов в csv файл
spinsTable = pd.DataFrame(spins, columns = [i for i in range(1,51)])
spinsTable.to_csv('band11.csv', index = False)